# M15 — Spectral & Temporal Representation Lab (Colab)

**Status: tasks T01–T03, T07–T08 authored (setup, data contract, legacy provenance, signal analytics, spectral resolution test, encoder benchmark, preservation, latent analytics, retrieval, G15) — never executed by the author.**

## A. Introduction

- **Objective.** Decide whether spectral (STFT / Welch / wavelet) and temporal (LSTM / GRU / TCN) representations of 10 Hz movement sequences are *reliable measurement instruments* — information-preserving and repeatable under comparable conditions — before any NFL prediction is attempted (M16–M18).
- **Hypotheses (H15.1–H15.5)** — all **[HYP]**: spectral summaries carry athlete information that survives length matching; learned encoders preserve physical structure better than PCA; latent dimensions are repeatable; cross-domain retrieval beats position-only retrieval; STFT and LSTM are complementary.
- **Data.** Portable M12 bundle: `speed, a_tan, a_n, yaw, dt, heading_valid` (scale `[10, 5, 5, 3, 0.1, 1]`), ≤ 64 frames at 10 Hz, Combine + NFL, 5 athlete-disjoint folds. No x/y, orientation, opponents or ball.
- **Dependencies.** Bundle only; exploratory STFT/LSTM artifacts only through the provenance audit ([EXP]).
- **Outputs.** `tables/{dataset, signal, spectral, preservation, latent, reliability, retrieval, legacy, synthesis}`, `features/*`, `checkpoints/*`, `embeddings/inductive/*`, `predictions/reconstruction/*`, figures `M15_*` (PNG/SVG/PDF + data), manifests, `reports/markdown/m15_handoff.md`.
- **Gate.** G15 is computed only from saved tables (section G15); every criterion defaults to NOT_EVALUATED.
- **Neural training** needs `run_training=True` **and** `allow_new_training=True` (GPU runtime). Without them only PCA is fitted.

Every step ends in an explicit status: COMPLETED · CACHED · BLOCKED_MISSING_INPUT · BLOCKED_PROVENANCE · BLOCKED_INSUFFICIENT_SAMPLE · FAILED_VALIDATION · FAILED · NOT_RUN.

## B. Setup

In [ ]:
# Setup 1 — mount Drive and install nflbowl from GitHub (the Colab PyTorch / CUDA build is never replaced)
from google.colab import drive
drive.mount('/content/drive')
REPO = 'git+https://github.com/mybenkhadda/NFL_bowl.git@main'
%pip install -q {REPO}
%pip install -q --upgrade --force-reinstall --no-deps {REPO}
%pip install -q ipywidgets umap-learn PyWavelets   # optional: a missing package only blocks the steps that need it

In [ ]:
# Setup 2 — version check (after a reinstall: Runtime → Restart session, then continue from here)
import nflbowl
print('nflbowl', nflbowl.__version__)
assert tuple(int(x) for x in nflbowl.__version__.split('.')[:2]) >= (0, 7), 'nflbowl < 0.7.0 is imported: restart the session and rerun from Setup 2'
from pathlib import Path
import numpy as np
import polars as pl
from IPython.display import Markdown, display
from nflbowl.devices import resolve_device
from nflbowl.lab import handoff
from nflbowl.lab import status as S
from nflbowl.lab.m15 import config as m15cfg, data as m15data, legacy as m15legacy, optional as m15opt, plots as m15plots
from nflbowl.lab.m15 import signals as m15sig, spectral as m15spec
display(m15opt.report())

In [ ]:
# Setup 3 — configuration (the ONLY cell to edit). Unknown keys raise; M12–M14 and the legacy folders are read only.
CFG = {
    'experiment_name': 'm15_spectral_temporal_v1',
    'bundle_dir': '/content/drive/MyDrive/nfl-big-data-bowl-2027/bundles/m12f_v1_clean',
    'drive_root': '/content/drive/MyDrive/NFLBowl',
    'legacy_m15_dirs': [],            # e.g. ['/content/drive/MyDrive/<exploratory M15 folder>'] — read only
    'legacy_split_manifests': [],     # optional explicit split files for those artifacts
    'device': 'auto',
    'seed': 42,
    'run_training': False, 'allow_new_training': False,   # Track B (T07) only; two opt-ins
}
cfg = m15cfg.resolve(CFG)
run = m15cfg.M15Run(cfg.run_dir, cfg)
env = handoff.environment(cfg.m13())
run.snapshot_config(cfg, env)
print('run dir           ', run.root)
print('config fingerprint', cfg.fingerprint())
print('device            ', resolve_device(cfg.device))
display(run.ledger.table())     # what earlier sessions already completed (resume)

## C. Data audit — E15.00 (M12 sequence-store adapter)

Opens and validates the bundle through `nflbowl.data.loading.open_bundle` (sizes, sha256 of small files, schemas), then `m15.data.audit_bundle`: fingerprint, channels, recorded scaling, shapes, row order, ids, domains, mask validity, physical plausibility on valid frames, heading convention, fold completeness / range / athlete-disjointness, linked athletes per fold, athletes-table contract. A failed contract check is **FAILED_VALIDATION** (tables are still written).

In [ ]:
# E15.00 — open, index, audit, persist (cached index reused when bundle + config fingerprints match)
def step_e15_00(ctx):
    data = m15data.prepare_m15_data(cfg, run, strict=False)
    ctx['outputs'] += ['tables/dataset/data_contract_audit', 'manifests/fold_assignments.parquet', 'manifests/dataset_fingerprint.json']
    m15data.assert_contract(data.audit)          # FAILED_VALIDATION after the evidence is saved
    return data

DATA = S.run_step(run, run.ledger, 'E15.00', step_e15_00, note='bundle audit + sequence index')
if DATA is None:
    display(run.ledger.table())
    raise SystemExit('E15.00 did not complete — read the reason above (and logs/e15_00_traceback.txt); later sections need the bundle.')
display(DATA.audit)
print('game key:', DATA.game_key_status)

In [ ]:
# Counts — sequences, athletes, trials per drill, linked cohort (re-read from the saved tables)
for name in ('valid_frames_by_domain', 'trials_per_drill', 'linked_cohort', 'counts_by_domain_kind_position'):
    display(Markdown(f'**{name}**')); display(run.frame(name, 'dataset'))
fa = DATA.folds
display(fa.group_by('fold', 'fold_source').agg(pl.len().alias('athletes'), (pl.col('has_combine') & pl.col('has_nfl')).sum().alias('both_domains')).sort('fold', 'fold_source'))

In [ ]:
# Figures — audit summary and valid-frame distributions (PNG / SVG / PDF + source data under figures/)
fig, dat = m15plots.audit_status_bar(DATA.audit)
run.figure(fig, 'M15_A0_01_audit_status', data=dat, question='Does the bundle satisfy the M15 data contract?', caption='Data-contract checks per status.')
fig, dat = m15plots.valid_frames_hist(DATA.index, cfg.max_frames)
run.figure(fig, 'M15_A0_02_valid_frames', data=dat, question='How long are Combine vs NFL sequences?',
           caption='Share of sequences by number of valid frames, per domain (Finding 2: NFL windows are shorter).')

## D. Exploratory analysis — sequence explorer

Physical-unit traces; invalid frames are gaps (never zeros); yaw is drawn only where heading is valid. Select athlete, domain and drill / demand kind; tick *save* to write the current view with its data.

In [ ]:
import matplotlib.pyplot as plt
idx = DATA.index
def show_sequences(nfl_id, domain, kind, max_n=6, save=False):
    sel = idx.filter((pl.col('nfl_id') == nfl_id) & (pl.col('domain') == domain) & ((pl.col('kind') == kind) if kind != '(all)' else pl.lit(True))).head(max_n)
    if not sel.height:
        print('no sequence for this selection'); return
    labels = [f"{r['kind']} · {r['seq_key']} · fold {r['fold']}" for r in sel.iter_rows(named=True)]
    fig, dat = m15plots.signal_panel(DATA.bundle, sel['idx'].to_numpy(), labels)
    if save:
        run.figure(fig, f'M15_A0_03_sequences_{nfl_id}_{domain}', data=dat, question='What do the raw movement signals look like?', caption='; '.join(labels))
    else:
        plt.show()

if m15opt.available('ipywidgets'):
    import ipywidgets as W
    ath = sorted(idx['nfl_id'].unique().to_list())
    w_ath = W.Dropdown(options=ath, description='athlete')
    w_dom = W.ToggleButtons(options=['combine', 'nfl'], description='domain')
    w_kind = W.Dropdown(options=['(all)'] + sorted(idx['kind'].unique().to_list()), description='kind')
    W.interact(show_sequences, nfl_id=w_ath, domain=w_dom, kind=w_kind, max_n=W.IntSlider(value=6, min=1, max=12), save=W.Checkbox(value=False))
else:
    first = int(idx.filter(pl.col('domain') == 'combine')['nfl_id'][0])
    show_sequences(first, 'combine', '(all)')

## E. Experiments

### E15.00-L — legacy (exploratory) STFT / LSTM artifact provenance

Classifies every file under `legacy_m15_dirs` as **verified_fold_safe**, **verified_exploratory_only** or **unknown_provenance** (rules in `nflbowl/lab/m15/legacy.py`). Sources are never modified. Only `verified_fold_safe` artifacts may enter supervised out-of-fold evaluation (`legacy.assert_fold_safe`). With no folder configured the step is **BLOCKED_MISSING_INPUT** — expected until the exploratory paths are supplied. Each experiment below is independently runnable and resumable (cached outputs → CACHED).

In [ ]:
def step_legacy(ctx):
    if not cfg.legacy_m15_dirs:
        raise S.ResearchBlocked(S.BLOCKED_MISSING_INPUT, 'no legacy_m15_dirs configured (Setup 3)', missing=['legacy_m15_dirs'])
    missing = [d for d in cfg.legacy_m15_dirs if not Path(d).exists()]
    if missing:
        raise S.ResearchBlocked(S.BLOCKED_MISSING_INPUT, 'legacy folder(s) not found on Drive', missing=missing)
    tab, recs = m15legacy.audit(cfg.legacy_m15_dirs, DATA.bundle, cfg.outer_folds, cfg.legacy_split_manifests, DATA.fold_of)
    ctx['outputs'] += run.table(tab, 'prototype_audit', 'legacy') + run.table(m15legacy.summary(recs), 'prototype_audit_summary', 'legacy')
    return tab, recs

LEGACY = S.run_step(run, run.ledger, 'E15.00-L', step_legacy, note='legacy STFT / LSTM provenance audit', requires=['E15.00'])
if LEGACY is not None:
    display(run.frame('prototype_audit_summary', 'legacy'))
    display(LEGACY[0].select('path', 'kind', 'provenance', 'reasons', 'n_rows', 'n_athletes', 'latent_dim'))
else:
    display(run.ledger.table().filter(pl.col('step') == 'E15.00-L'))

## E15.A1 — time-domain features and validity diagnostics

Per sequence, on valid frames only: max / q90 / mean / entry speed (yd/s), max / min a_tan (yd/s²), braking- and accelerating-frame shares (|a_tan| ≥ 1 yd/s²), max |a_n|, jerk RMS (yd/s³, frame pairs inside one gap-free segment), derivative consistency (RMSE of Δv/Δt vs a_tan), heading-valid share, valid frames and seconds. Cached in `features/signal/sequence_time_features` (reused when bundle and config fingerprints match). Validity tables by domain, kind and position.

In [ ]:
def step_a1(ctx):
    return m15sig.run_a1(DATA, cfg, run, ctx)

FEAT = S.run_step(run, run.ledger, 'E15.A1', step_a1, note='per-sequence time-domain features + validity tables', requires=['E15.00'])
if FEAT is not None:
    for name in ('validity_by_domain', 'validity_by_domain_kind', 'validity_by_domain_position'):
        display(Markdown(f'**{name}**')); display(run.frame(name, 'signal'))
else:
    display(run.ledger.table().filter(pl.col('step') == 'E15.A1'))

## E15.A2 — Combine vs NFL under successive matching (Finding 2)

The reported medians [EXP] (max speed 7.74 vs 5.89 yd/s, braking-frame fraction 0.125 vs 0.345, 62 vs 43 valid frames) are recomputed in four steps: **raw** (all sequences) → **athlete_weighted** (median of athlete medians) → **length_matched** (Combine cropped to contiguous gap-free windows with NFL-distributed lengths) → **cem** (additionally matched on entry speed, length and position). `share_of_raw_remaining` shows how much of the gap survives. Then a descriptive Combine-drill × NFL-demand profile. Braking frame := a_tan ≤ −1 yd/s² (M14 threshold); the brief's exact definition is unknown, so values may differ from it. These are context differences, never evidence of athletes changing.

In [ ]:
def step_a2(ctx):
    if FEAT is None:
        raise S.ResearchBlocked(S.BLOCKED_MISSING_INPUT, 'E15.A1 feature table not available in this session', missing=['E15.A1'])
    return m15sig.run_a2(DATA, FEAT, cfg, run, ctx)

A2 = S.run_step(run, run.ledger, 'E15.A2', step_a2, note='length / entry-speed / athlete-matched domain contrasts', requires=['E15.A1'])
if A2 is not None:
    display(A2['steps'].select('metric', 'unit', 'step', 'nfl_median', 'combine_median', 'diff_nfl_minus_combine', 'lo', 'hi', 'share_of_raw_remaining',
                               'athletes_nfl', 'athletes_combine'))
    display(Markdown('**CEM balance (standardised mean differences before / after)**')); print(A2['balance'])
    display(Markdown('**Athlete-weighted profile per domain × kind**')); display(A2['profile'])
else:
    display(run.ledger.table().filter(pl.col('step') == 'E15.A2'))

In [ ]:
# E15.A2 figures — contrast waterfall and Combine-drill × NFL-demand matrices
if A2 is not None:
    fig, dat = m15plots.contrast_waterfall(A2['steps'])
    run.figure(fig, 'M15_A2_01_contrast_waterfall', data=dat, question='How much of the Combine / NFL difference is context, weighting and length?',
               caption='NFL − Combine median difference with 95 % athlete-bootstrap CI at each matching step.')
    for m in ('max_speed', 'braking_frac', 'max_atan'):
        fig, dat = m15plots.kind_heatmap(A2['kind_contrast'], m)
        run.figure(fig, f'M15_A2_02_kind_matrix_{m}', data=dat, question='Which Combine drills resemble which NFL demands?',
                   caption=f'Athlete-weighted median difference (NFL demand − Combine drill) of {m}; descriptive, kinds are not matched pairs.')

## E15.A3 — spectral features at controlled resolution and the resolution-artifact test

Welch PSD (Hann, 50 % overlap) on the longest gap-free segment, for speed and a_tan, nperseg ∈ `welch_nperseg` (Δf = 10 / nperseg Hz) and detrend ∈ `detrend`; segments shorter than nperseg give no estimate (never padded). Features exclude the DC bin. **Artifact test** per feature × configuration × domain: Spearman with segment length, share of centroids pinned to the first non-zero bin, AUC vs a ramp (trend-only) surrogate and vs a white-noise surrogate. Estimable := |ρ_len| ≤ 0.3, pinned ≤ 0.5, separation AUC ≥ 0.7. The domain contrast is recomputed on equal-length crops of both domains. The reported centroid ≈ 0.312 Hz [EXP] is exactly 10/32 Hz — this section tests whether that is the estimator rather than the athlete.

In [ ]:
def step_a3(ctx):
    return m15spec.run_a3(DATA, cfg, run, ctx)

A3 = S.run_step(run, run.ledger, 'E15.A3', step_a3, note='Welch features + resolution-artifact test + length-matched contrast', requires=['E15.00'])
if A3 is not None:
    art = A3['artifact']
    display(Markdown('**Estimable share per configuration and domain**'))
    display(art.group_by('channel', 'nperseg', 'detrend', 'domain').agg(pl.col('estimable').mean().alias('share_estimable'), pl.len().alias('n_features'))
            .sort('channel', 'nperseg', 'detrend', 'domain'))
    display(Markdown('**Centroid: pinned share and length dependence**'))
    display(art.filter(pl.col('feature') == 'centroid').select('channel', 'nperseg', 'detrend', 'domain', 'median', 'first_bin_hz', 'pinned_share', 'rho_len',
                                                               'auc_vs_ramp', 'auc_vs_noise', 'estimable'))
    display(Markdown('**Length-matched NFL − Combine differences (athlete-weighted, 95 % CI)**')); display(A3['contrast'])
else:
    display(run.ledger.table().filter(pl.col('step') == 'E15.A3'))

In [ ]:
# E15.A3 figures — resolution artifact, equal-resolution PSD, spectrogram examples
if A3 is not None:
    dom = dict(zip(DATA.index['idx'].to_list(), DATA.index['domain'].to_list()))
    for nps in cfg.welch_nperseg:
        fig, dat = m15plots.centroid_vs_length(A3['real'], dom, cfg.sampling_hz, 'speed', int(nps), 'linear', seed=cfg.seed)
        run.figure(fig, f'M15_A3_01_centroid_vs_length_n{nps}', data=dat, question='Is the speed spectral centroid a resolution artifact?',
                   caption=f'Speed centroid vs analysed segment length; dashed = first non-zero bin {cfg.sampling_hz / nps:.4f} Hz (nperseg {nps}).')
    for ch in m15spec.CHANNELS:
        fig, dat = m15plots.psd_by_domain(DATA.bundle, DATA.index, cfg, ch, int(max(cfg.welch_nperseg)), seed=cfg.seed)
        run.figure(fig, f'M15_A3_02_psd_by_domain_{ch}', data=dat, question='Do Combine and NFL spectra differ at equal resolution?',
                   caption=f'{ch} Welch PSD median and IQR on equal-length crops (descriptive; differences are not athlete traits).')
    fig, dat = m15plots.spectrogram_gallery(DATA.bundle, DATA.index, cfg, seed=cfg.seed)
    run.figure(fig, 'M15_A3_03_spectrogram_gallery', data=dat, question='What do STFT magnitudes look like per drill / demand?',
               caption='Speed |STFT| for one example per kind; invalid frames zero after mean removal (encoder-input convention).')

## E15.A4 — spectral reliability, perturbation sensitivity and measurement status

ICC(1,1) per Combine drill (≥ 30 athletes with ≥ 2 trials) and across NFL windows after kind-centring; NFL split-half (season split when ≥ 2 seasons, else random halves — labelled); M14 perturbations (speed noise 0.05 / 0.10 yd/s, 3-frame smoothing) on a seeded 150-athlete Combine sample. Status = M14 rule: **validated** needs estimable + ICC ≥ 0.6 (lower CI ≥ 0.4) + min perturbation ρ ≥ 0.8; missing evidence is `not_evaluated`, never validated.

In [ ]:
def step_a4(ctx):
    if A3 is None:
        raise S.ResearchBlocked(S.BLOCKED_MISSING_INPUT, 'E15.A3 outputs not available in this session', missing=['E15.A3'])
    return m15spec.run_a4(DATA, A3['real'], A3['artifact'], cfg, run, ctx)

A4 = S.run_step(run, run.ledger, 'E15.A4', step_a4, note='spectral reliability + sensitivity + measurement status', requires=['E15.A3'])
if A4 is not None:
    st = A4['status']
    display(st.group_by('measurement_status').len().sort('measurement_status'))
    display(st.sort('channel', 'nperseg', 'detrend', 'feature'))
    display(Markdown('**NFL split-half reliability**')); display(A4['nfl_split'].sort('channel', 'nperseg', 'detrend', 'feature'))
else:
    display(run.ledger.table().filter(pl.col('step') == 'E15.A4'))

In [ ]:
# E15.A4 figures — reliability forests for the primary configuration (Welch nperseg = max, linear detrend)
if A4 is not None:
    n_primary = int(max(cfg.welch_nperseg))
    for ch in m15spec.CHANNELS:
        fig, dat = m15plots.icc_forest(A4['reliability'], ch, n_primary, 'linear')
        run.figure(fig, f'M15_A4_01_icc_forest_{ch}', data=dat, question='Are spectral features repeatable within athlete?',
                   caption=f'ICC(1,1) with athlete-bootstrap 95 % CI per feature and scope ({ch}, nperseg {n_primary}, linear detrend); dashed 0.4 / 0.6.')

## E15.B1 — encoder benchmark under an equal budget (T07)

`cfg.architectures × latent_dims × train_seeds × folds`. PCA is always fitted (closed form). Learned encoders (STFT-CNN-AE, BiLSTM/BiGRU-AE, TCN-AE on the m11 `ConvEncoder`, masked TCN) train only with **both** `run_training=True` and `allow_new_training=True` in Setup 3 (GPU runtime); otherwise previously finished checkpoints are reused and unfinished ones are NOT_RUN. Every fold model is trained on athletes of the other folds only (early stopping on a validation share of those athletes); held-out athletes are only embedded. Resumable from `last.pt`; finished fits are never retrained. Widths are chosen to match the TCN reference parameter budget (± `param_band`).

In [ ]:
from nflbowl.lab.m15 import pipeline as m15pipe
from nflbowl.lab.m16 import retrieval as m16ret
DEVICE = resolve_device(cfg.device)

def step_b(ctx):
    return m15pipe.run_b(DATA, cfg, run, ctx, DEVICE)

TRAIN = S.run_step(run, run.ledger, 'E15.B1', step_b, note='encoder grid (PCA always; learned models need two opt-ins)', requires=['E15.00'])
if TRAIN is not None:
    display(run.frame('parameter_budget', 'latent'))
    display(TRAIN)

In [ ]:
if TRAIN is not None and run.has('training_curves', 'latent'):
    fig, dat = m15plots.training_curves(run.frame('training_curves', 'latent'))
    run.figure(fig, 'M15_B_01_training_curves', data=dat, question='Did every encoder train comparably?', caption='Validation-athlete loss vs step (equal budget).')

## E15.C1–C2 — information preservation and robustness (held-out athletes)

Errors on frames valid in the original only (yaw on heading-valid frames): physical RMSE per channel, peak errors, M14 event F1 and braking-onset timing, Welch band-share error, derivative consistency; STFT models report spectrogram loss only (no time-domain reconstruction — never imputed). Robustness: block dropout 10/20/30 % and truncation to NFL length quantiles → error on original valid frames, embedding drift, self-retrieval.

In [ ]:
def step_c(ctx):
    return m15pipe.run_c(DATA, cfg, run, ctx, DEVICE, n_per_domain=int(cfg.extra.get('preservation_rows_per_domain', 400)))

PRES = S.run_step(run, run.ledger, 'E15.C', step_c, note='preservation (valid frames only) + robustness', requires=['E15.B1'])
if PRES is not None:
    display(PRES['summary'])
    display(Markdown('**Paired vs PCA (model − PCA; error metrics: negative = better)**')); display(PRES['paired'])
    display(Markdown('**Robustness (first selected fold)**')); display(PRES['robustness'])

In [ ]:
# E15.C figures — preservation matrix and one reconstruction example per domain
if PRES is not None:
    fig, dat = m15plots.preservation_matrix(PRES['summary'])
    run.figure(fig, 'M15_C_01_preservation_matrix', data=dat, question='Which encoder preserves physical structure?', caption='Held-out medians per architecture and domain.')
    from nflbowl.lab.m15 import training as m15tr
    X_, M_ = m15tr.arrays(DATA)
    for dom in ('combine', 'nfl'):
        rows_ = PRES['rows'].filter((pl.col('domain') == dom) & (pl.col('fold') == cfg.selected_folds()[0]))
        if not rows_.height:
            continue
        i = int(rows_['idx'][0])
        recons = {}
        for s in m15pipe.finished_specs(cfg, run):
            if s.fold == cfg.selected_folds()[0]:
                mdl = m15tr.load_model(s, cfg, run, DEVICE)
                if getattr(mdl, 'domain', 'time') == 'time':
                    recons[f'{s.arch} d{s.d}'] = m15tr.reconstruct_rows(mdl, X_, M_, np.array([i]), DEVICE)[0]
        if recons:
            fig, dat = m15plots.reconstruction_overlay(X_[i], M_[i], recons, f'{dom} sequence row {i} (held-out athlete)')
            run.figure(fig, f'M15_C_02_overlay_{dom}', data=dat, question='What does each model keep or lose?', caption='Original (black) vs reconstructions on valid frames.')

## E15.D1–D3 — latent-space analytics (held-out rows only)

Geometry (effective rank, collapse, norms, within/between-athlete variance), fold-safe probes (domain AUC, position balanced accuracy), per-dimension ICC across repeated Combine attempts, within-position repeat-attempt AUC, cross-seed Procrustes stability, and position-controlled correlations with simple forty features (a dimension may be *named* only under the stated rule).

In [ ]:
def step_d(ctx):
    return m15pipe.run_d(DATA, cfg, run, ctx, feat=FEAT, icc_boot=0)

LAT = S.run_step(run, run.ledger, 'E15.D', step_d, note='latent geometry, probes, ICC, repeat AUC, seed stability, interpretation', requires=['E15.B1'])
if LAT is not None:
    display(LAT['geometry'])
    display(LAT['probes'])
    display(LAT['icc_summary'])
    display(LAT['repeat_auc'])
    display(LAT['seed_stability'])
    display(Markdown('**Allowed dimension labels (|ρ| ≥ 0.5, CI beyond 0.3, sign-consistent across seeds)**')); display(LAT['labels'])

In [ ]:
# E15.D figures — latent repeatability and projections (projections are descriptive only)
if LAT is not None:
    if LAT['dim_icc'].height:
        fig, dat = m15plots.latent_icc_strip(LAT['dim_icc'])
        run.figure(fig, 'M15_D_01_latent_icc', data=dat, question='Are latent dimensions repeatable across Combine attempts?', caption='Per-dimension ICC(1,1); dashed 0.4 / 0.6.')
    if LAT['projection'].height:
        first = LAT['projection'].select('arch', 'd').unique().rows()[0]
        proj = LAT['projection'].filter((pl.col('arch') == first[0]) & (pl.col('d') == first[1]))
        for by in ('domain', 'position_group', 'kind'):
            fig, dat = m15plots.projection_scatter(proj, by, 'umap' if 'umap1' in proj.columns else 'pc')
            run.figure(fig, f'M15_D_02_projection_{first[0]}_{by}', data=dat, question='How is the latent space organised?',
                       caption='Descriptive projection of held-out embeddings; not evidence of predictive value.')

## E15.E1–E3 — preliminary transfer: five-fold cross-domain retrieval

Held-out athletes with both domains; query = Combine athlete mean, gallery = NFL means in the pool (**all**, **same_position**, **twins** = same position and |Δforty| ≤ 0.03 s). Arms: unaligned, Procrustes and InfoNCE heads (fitted on TRAINING athletes only), random; F0 engineered features; STFT ⊕ LSTM concatenation. The prototype's 7/80 top-1 [EXP] is re-read against uniform chance and the position-only expectation (≈ 1/16 with five positions). Exploratory here; the confirmatory test is M16 E16.B1.

In [ ]:
def step_e(ctx):
    forty = m16ret.forty_map(DATA.bundle.athletes)
    e0 = None
    if cfg.extra.get('f0_reference', True):
        exp = handoff.open_experiment(cfg.m13(), run)          # F0 engineered features (cached by the experiment); no training
        e0 = exp.features()
    return m15pipe.run_e(DATA, cfg, run, ctx, forty=forty, e0=e0)

E = S.run_step(run, run.ledger, 'E15.E', step_e, note='5-fold cross-domain retrieval with controls', requires=['E15.B1'])
if E is not None:
    display(E['summary'].sort('pool', 'rep', 'arm'))
    display(Markdown('**Paired MRR differences**')); display(E['paired'])
    display(Markdown('**Position-only expectation per fold**')); display(E['position_chance'])
    display(Markdown('**Alignment damage check (E15.E3)**')); display(E['damage'])

In [ ]:
# E15.E figures — retrieval against uniform chance and the position-only expectation
if E is not None and E['summary'].height:
    for pool in ('same_position', 'all', 'twins'):
        if E['summary'].filter(pl.col('pool') == pool).height:
            fig, dat = m15plots.retrieval_bars(E['summary'], E['position_chance'], pool)
            run.figure(fig, f'M15_E_01_retrieval_{pool}', data=dat, question='Does a Combine embedding identify the same athlete in NFL windows?',
                       caption=f'Top-1 and MRR in the {pool} pool; ticks = uniform chance; dashed (all pool) = position-only expectation.')
    display(E['prototype'])

In [ ]:
# G15 — gate from the saved tables (never from code or configuration alone)
def step_g15(ctx):
    return m15pipe.run_gate(cfg, run, ctx)

G15 = S.run_step(run, run.ledger, 'G15', step_g15, note='representation readiness gate')
if G15 is not None:
    display(G15.filter(pl.col('criterion') == 'G15 decision').select('arch', 'd', 'status'))
    fig, dat = m15plots.gate_matrix(G15, 'G15')
    run.figure(fig, 'M15_G_01_gate_matrix', data=dat, question='Which representations may advance to M16 / M18?', caption='G15 criteria per representation.')

## F. Results, gate status and next steps

Read from the saved tables only:
- **E15.A2** — Combine/NFL differences after weighting, length and entry-speed matching (`tables/signal/domain_contrast_steps`).
- **E15.A3/A4** — spectral measurement status (`tables/spectral/measurement_status`); only `validated` features may enter M16 as predictors.
- **E15.B–E** — preservation vs PCA, latent repeatability, retrieval vs position-only chance.
- **G15** (`tables/synthesis/m15_gate`) — only `GO` representations (and `GO_DESCRIPTIVE` when M16 `allow_descriptive` is set, labelled) advance to M16 E16.B1 and M18 B4–B6. With learned training disabled, only PCA is evaluated and the learned rows read NOT_RUN / NOT_EVALUATED — never PASS.

Next: M16 (targets, reliability, baselines, confirmatory retrieval, leakage audit) → M17 → M18.

In [ ]:
# Summary of computed decisions (the gate table is written only by the G15 step above)
if run.has('m15_gate', 'synthesis'):
    g = run.frame('m15_gate', 'synthesis')
    display(g.filter(pl.col('criterion') == 'G15 decision').select('arch', 'd', 'status'))
else:
    print('G15: NOT_EVALUATED (the gate step did not complete this session)')
if run.has('measurement_status', 'spectral'):
    display(Markdown('**Spectral measurement status (computed)**'))
    display(run.frame('measurement_status', 'spectral').group_by('measurement_status').len().sort('measurement_status'))
display(run.ledger.table())

## G. Persistence and hand-off

In [ ]:
run.write_status()
run.save_markdown(run.handoff_markdown(), 'm15_handoff')
for rel in ('manifests/m15_status.json', 'manifests/m15_run_status.json', 'manifests/dataset_fingerprint.json', 'manifests/fold_assignments.parquet',
            'features/signal', 'features/spectral', 'checkpoints', 'embeddings/inductive', 'predictions/reconstruction', 'tables/dataset', 'tables/signal',
            'tables/spectral', 'tables/preservation', 'tables/latent', 'tables/reliability', 'tables/retrieval', 'tables/synthesis', 'tables/legacy',
            'figures/png', 'figures/data', 'reports/markdown'):
    p = run.root / rel
    print(('✓ ' if p.exists() else '· ') + str(p))
print('\nSEND BACK: manifests/stages.json, manifests/m15_status.json, logs/*_traceback.txt, tables/dataset/data_contract_audit.csv,')
print('           tables/signal/domain_contrast_steps.csv, tables/spectral/artifact_test.csv, tables/spectral/measurement_status.csv,')
print('           tables/latent/training_status.csv, tables/preservation/paired_vs_pca.csv, tables/reliability/latent_icc_summary.csv,')
print('           tables/retrieval/summary.csv, tables/retrieval/paired.csv, tables/synthesis/m15_gate.csv')